# 2. Automatic Jev input design

Inspect deterministic task questions, full label-free states, and workload **before spending money**. These are provider-neutral intended requests. Notebook 3 also shows the documentation-verified wire mapping. No live client exists.

## 1. Inputs and scope

This notebook reuses the full audit through the same source function as Notebook 1 and can rebuild it independently. No folds, fitted models, target correlations or benchmark metrics are defined.

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.config import load_config, save_resolved_config
from src.utils.notebook_display import display_local, display_json
from src.utils.provenance import provenance
from src.data.audit import audit_collection
from src.visualize import figures, style
style.apply()
cfg = load_config()
from src.jev.templates import BINARY, MULTICLASS, REGRESSION, REGRESSION_LABELS
from src.jev.inspection import representative_requests, design_summary
from src.jev.cost import estimate_workload, workload_totals
from src.jev.cache import ResponseCache, dry_run
from tempfile import TemporaryDirectory

NOTEBOOK = "02_jev_input_design"
from src.utils.notebook_report import begin_report, report_section, finish_report
begin_report()
report_section('1. Inputs and scope')
save = figures.FigureSaver(NOTEBOOK)
config_hash = save_resolved_config(cfg, NOTEBOOK)
audit = audit_collection(cfg)
display_json("Execution provenance", provenance())


## 2. Deterministic task templates

The single source of wording is `src/jev/templates.py`. No LLM generates questions. The binary positive class is the last label in canonical JSON order; this is a reproducible orientation, **not a claim about domain meaning**. The other label is the comparison class. Multiclass retains the complete allowed vocabulary. Regression has nine ordered directional choices, with zero meaning no directional evidence; converting eventual outputs into numeric features remains open.

In [ ]:
report_section('2. Deterministic task templates')
display_json("Central question templates and regression scale", {"binary": BINARY, "multiclass": MULTICLASS, "regression": REGRESSION, "regression_scale": REGRESSION_LABELS})


## 3. Representative request states

The same builder handles all datasets. Examples cover binary, multiclass and regression, with one or multiple text columns. Read each text column separately, and all available text columns together. Non-text columns never enter Jev. Metadata supplies the target name, task type and class names; true row targets and label statistics are excluded.

Empty/null/whitespace-only text is skipped. A joint request omits missing fields and is skipped only when every text field is empty. Skipped inputs will become missing numeric features for TabPFN. No nonempty text is truncated. Identical requests share cached responses.


In [ ]:
report_section('3. Representative request states')
requests = representative_requests(audit, cfg)
for request in requests:
    origin = request.to_dict()["provenance"]
    title = f"{origin['dataset_id']} | {origin['representation']} | {', '.join(origin['text_columns'])}"
    display_json(title, request.to_dict())


## 4. Workload and configurable cost

Request counts are **logical slots before deduplication**, after skipping empty inputs. Before skipping empty inputs, for `n` rows and `t` text columns: per-column = `n × t`, joint = `n`, combined = `n × (t + 1)`. Single-text per-column and joint requests have identical content and will share a cached response. Other duplicates may also save work; savings are not guessed.

Approximate input tokens include the exact compact JSON of the intended task/state/question, including class choices. They are calculated from actual per-row encoded lengths, not a sampled mean. The calculation is tested against individual request serialisation.

Set `cost.input_price_per_million` in `config/exploration/audit.yaml` or `JEV_INPUT_PRICE_PER_MILLION`. The dated reference is **USD 0.042 per million input tokens**, from https://docs.typesafe.ai/models (checked 2026-09-29). Recheck before paid execution. `cost_per_unit_price` gives the coefficient to multiply by the verified price per million input tokens. This is a configurable pricing assumption, not an account quote. Output tokens are currently free; unknown provider wrappers and retries remain excluded; optional overhead is only a sensitivity assumption. This is not a quote or billed-token forecast.

In [ ]:
report_section('4. Workload and configurable cost')
estimates = estimate_workload(audit, cfg)
totals = workload_totals(estimates)
display_local(totals)
display_local(estimates[(estimates["mode"] == cfg["jev"]["representation"]) & (estimates.include_non_text_features == cfg["jev"]["include_non_text_features"])][["dataset", "requests", "input_tokens_approx", "max_request_tokens_approx", "input_cost_estimate"]])


## 5. Cache and open choices

This demonstration uses a temporary directory and writes only a mock acknowledgement with `sent: false` and no probabilities. A second lookup resumes from the same transactional SQLite cache. Mock entries are separate from any future verified-response entries. Keys use exact payload and version/configuration; provenance retains dataset, positional row ID, text columns, source hashes, code/config hashes and timestamps. The temporary demo database is deleted automatically at the end of the cell. A future paid-response cache will live under `data/jev_cache/`, outside disposable output/processed cleanup; these notebooks do not create it.

Before enabling real calls, review `docs/DESIGN_NOTES.md`: The documented API shape and current model/pricing are recorded in feature-creation config. Live behaviour, output selection, context handling, dataset semantics and evaluation remain to be reviewed. Concurrent paid-call reservation/recovery and provider idempotency must be implemented with the verified API; the present interface deliberately has no network transport.

In [ ]:
report_section('5. Cache and open choices')
with TemporaryDirectory(prefix="jevpfn-cache-demo-") as temporary_cache:
    cache = ResponseCache(Path(temporary_cache) / "responses.sqlite3")
    first_lookup = dry_run(requests[0], cache)
    second_lookup = dry_run(requests[0], cache)
    assert second_lookup["cache_hit"]
    assert second_lookup["response"]["sent"] is False
    display_json("Temporary mock cache demonstration", {"first": first_lookup, "second": second_lookup})


## Summary

The final cell prints every displayed detail in section order and saves it in the ignored output folder.

In [ ]:
finish_report(NOTEBOOK, design_summary(audit, requests, totals, cfg) + "\n" + save.summary())
